In [1]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import random
import os
import torch
from torch import nn
from torch.nn import functional as F
from collections import deque

In [ ]:
# Critic Network
class Critic(nn.Module):
    def __init__(self, num_states, num_actions):
        super().__init__()
        self.fc1 = nn.Linear(num_states, 64)
        self.fc2 = nn.Linear(64, 64)
        self.out = nn.Linear(64, num_actions)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.out(x)

# Actor Network
class Actor(nn.Module):
    def __init__(self, num_states, num_actions):
        super().__init__()
        self.fc1 = nn.Linear(num_states, 64)
        self.fc2 = nn.Linear(64, 64)
        self.out = nn.Linear(64, num_actions)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.out(x)

# memory for experience replay
class ReplayMemory():
    def __init__(self, maxlen):
        self.memory = deque([], maxlen=maxlen)

    def append(self, transition):
        self.memory.append(transition)

    def sample(self, sample_size):
        return random.sample(self.memory, sample_size)

    def __len__(self):
        return len(self.memory)

# the observation is a numpy array, net input is a float32 tensor
def state_to_nn_input(state):
    return torch.tensor(state, dtype=torch.float32)

# one training step of the critic net on a batch from memory
def optimize(discount_factor_g, batch, critic_nn, critic_target_nn, loss_fn, optimizer):

    states = torch.stack([state_to_nn_input(s) for s, a, ns, r, t in batch]) # (batch_size, num_states)
    actions = torch.tensor([a for s, a, ns, r, t in batch], dtype=torch.int64).unsqueeze(-1) # (batch_size, 1) for gather()
    next_states = torch.stack([state_to_nn_input(ns) for s, a, ns, r, t in batch]) # (batch_size, num_states)
    rewards = torch.tensor([r for s, a, ns, r, t in batch], dtype=torch.float32) # (batch_size,)
    terminated_flags = torch.tensor([float(t) for s, a, ns, r, t in batch], dtype=torch.float32) # (batch_size,)

    # Q(S, A) for the current state and the actions that were taken from that state
    # gather() picks the Q value of the action that was taken
    current_q_values = critic_nn(states).gather(1, actions).squeeze(-1)

    with torch.no_grad(): # gradients are not needed for the critic_target_nn as updating only critic_nn
        # max Q value for the next state from the target net
        next_q_values = critic_target_nn(next_states).max(1).values
        # the value of a terminal state is 0
        next_q_values[terminated_flags == 1] = 0.0

        # 1 step TD target = reward + gamma * max_a Q_target(S', a)
        target_q_values = rewards + (discount_factor_g * next_q_values)

    loss = loss_fn(current_q_values, target_q_values)

    # learn
    optimizer.zero_grad()
    loss.backward()
    # clip gradients to prevent exploding gradients
    torch.nn.utils.clip_grad_norm_(critic_nn.parameters(), max_norm=1.0)
    optimizer.step()

In [ ]:
# Soft Actor Critic
env = gym.make("CartPole-v1", render_mode=None)

# env parameters
num_states = env.observation_space.shape[0] # (cart x, cart x_dot, pole theta, pole theta_dot)
num_actions = env.action_space.n # (left or right)

# hyperparameters
num_episodes = 5000 # num of epi to train
discount_factor_g = 0.99

# all networks
actor_nn = Actor(num_states=num_states, num_actions=num_actions)
critic_nn = Critic(num_states=num_states, num_actions=num_actions)
critic_target_nn = Critic(num_states=num_states, num_actions=num_actions)

step_count = 0
network_sync_rate = 500 # after 500 steps copy the critic net to the critic target net

# training parameters
batch_size = 64
learning_rate = 0.001
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(critic_nn.parameters(), lr=learning_rate)

# memory for recording transitions
replay_memory_size = 5000
memory = ReplayMemory(replay_memory_size)

# to track reward per episode
reward_per_episode = np.zeros(num_episodes)


In [ ]:
for i_episode in range(num_episodes):

    state, info = env.reset()
    terminated = False
    truncated = False
    episode_reward = 0

    while (not terminated and not truncated):
        with torch.no_grad():
            actor_input = state_to_nn_input(state).unsqueeze(0)
            actor_output = actor_nn(actor_input)
            action = torch.distributions.Categorical(logits=actor_output).sample().item()

        next_state, reward, terminated, truncated, info = env.step(action)

        # Termination: Pole Angle is greater than ±12°
        # Termination: Cart Position is greater than ±2.4
        # Truncation: Episode length is greater than 500
        # save terminated not truncated, at 500 steps the pole is still up so S' still has value
        memory.append((state, action, next_state, reward, terminated))

        # optimize at every step, taking a sample from memory
        # start this when memory size > batch size
        if len(memory) > batch_size:
            batch = memory.sample(batch_size)
            optimize(discount_factor_g, batch, critic_nn, critic_target_nn, loss_fn, optimizer)
            # (TODO)we need to optimize actor, update previous name to optimize_critic

        # copy the critic net to the critic target net
        if step_count % network_sync_rate == 0:
            critic_target_nn.load_state_dict(critic_nn.state_dict())

    # keep track of reward at every episode
    reward_per_episode[i_episode] = episode_reward

    # progress report
    if (i_episode + 1) % 50 == 0:
        avg_reward = np.mean(reward_per_episode[max(0, i_episode-49):(i_episode+1)])
        print(f"Episode {i_episode+1}/{num_episodes} | Avg Reward (last 50): {avg_reward:.2f}")

env.close()